# 04 — Model tables

## Manuscript crosswalk

- **Methods:** Analysis pipeline overview; eligible session-pair construction; repertoire-distance aggregation; metric-wise standardization; model-data preparation.
- **Results:** 331 sequence comparisons and 431 call comparisons, expanded to four metric rows per comparison.
- **Figure:** Figure S1 (analytical workflow).

This notebook audits the exact long-form tables supplied to the Bayesian models. It performs only inexpensive checks and joins. It never calculates a distance matrix or fits a model.

## From repertoires to model rows

Within each focal pair, stage, and audience context, every eligible repertoire from one partner is paired with every eligible repertoire from the other partner. Each unique repertoire pair receives one distance for each of four metrics. Raw distances are standardized within metric across all eligible pairs, stages, and contexts for that structural level.

Canonical files:

- `data/derived/sequence_repertoire_distances.csv`
- `data/derived/call_repertoire_distances.csv`
- `data/derived/model_sequence.csv`
- `data/derived/model_call.csv`

See [the data dictionary](../docs/data_dictionary.md) for field definitions.

In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter inside a clone containing README.md and data/."
    )


ROOT = find_repo_root()
EXPENSIVE_STEPS = {
    "recompute_sequence": False,
    "recompute_dtw": False,
    "retrain_vae": False,
    "refit_models": False,
}
if any(EXPENSIVE_STEPS.values()):
    raise RuntimeError("Reader-facing notebooks are cached-only; use explicit full mode instead.")

pd.DataFrame({"step": EXPENSIVE_STEPS.keys(), "enabled": EXPENSIVE_STEPS.values()})

In [ ]:
MANIFEST_PATHS = (
    ROOT / "data/processed/manifest.json",
    ROOT / "data/cache/manifest.json",
    ROOT / "data/derived/manifest.json",
    ROOT / "results/manifest.json",
)
FIGURE_PROVENANCE_PATH = ROOT / "results/figures/figure_provenance.csv"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_manifests() -> list[tuple[Path, dict]]:
    manifests = []
    for path in MANIFEST_PATHS:
        if path.is_file():
            with path.open(encoding="utf-8") as handle:
                manifests.append((path, json.load(handle)))
    if not manifests:
        raise FileNotFoundError("No JSON provenance manifest was found.")
    return manifests


def iter_records(payload: dict):
    for section_name in ("artifacts", "sources", "inputs", "files"):
        section = payload.get(section_name, {})
        if isinstance(section, dict):
            for key, value in section.items():
                record = value if isinstance(value, dict) else {"sha256": value}
                yield str(record.get("path", key)), record
        elif isinstance(section, list):
            for record in section:
                if isinstance(record, dict) and record.get("path"):
                    yield str(record["path"]), record


def record_sha256(record: dict) -> str | None:
    value = record.get("sha256") or record.get("checksum_sha256")
    if value:
        return str(value).removeprefix("sha256:")
    checksum = record.get("checksum")
    if isinstance(checksum, str):
        return checksum.removeprefix("sha256:")
    if isinstance(checksum, dict) and checksum.get("algorithm", "").lower() == "sha256":
        return checksum.get("value")
    return None


MANIFESTS = load_manifests()
FIGURE_PROVENANCE = (
    pd.read_csv(FIGURE_PROVENANCE_PATH) if FIGURE_PROVENANCE_PATH.is_file() else pd.DataFrame()
)


def registered_artifact(relative_path: str) -> Path:
    path = ROOT / relative_path
    if not path.is_file():
        raise FileNotFoundError(
            f"Required artifact is missing: {relative_path}. Cached mode will not recompute it."
        )
    normalized = Path(relative_path).as_posix()
    for manifest_path, payload in MANIFESTS:
        for recorded_path, record in iter_records(payload):
            recorded = Path(recorded_path)
            same_path = (recorded.resolve() == path.resolve()) if recorded.is_absolute() else (recorded.as_posix().lstrip("./") == normalized)
            if same_path:
                expected = record_sha256(record)
                if not expected:
                    raise RuntimeError(f"No SHA-256 for {relative_path} in {manifest_path}.")
                if sha256_file(path).lower() != expected.lower():
                    raise RuntimeError(f"Checksum mismatch for {relative_path}.")
                return path
    if not FIGURE_PROVENANCE.empty and {"path", "sha256"}.issubset(FIGURE_PROVENANCE.columns):
        match = FIGURE_PROVENANCE.loc[FIGURE_PROVENANCE["path"] == normalized]
        if len(match) == 1 and sha256_file(path).lower() == str(match.iloc[0]["sha256"]).lower():
            return path
    raise RuntimeError(f"{relative_path} is not registered in a provenance manifest.")

## Load and validate canonical schemas

The two repertoire-distance tables contain raw distances. The two model tables retain those values and add the within-metric standardized response and centered stage coding.

In [ ]:
paths = {
    "sequence_distances": "data/derived/sequence_repertoire_distances.csv",
    "call_distances": "data/derived/call_repertoire_distances.csv",
    "sequence_model": "data/derived/model_sequence.csv",
    "call_model": "data/derived/model_call.csv",
    "sequence_inventory": "data/cache/sequence_session_inventory.csv",
    "call_inventory": "data/cache/call_session_inventory.csv",
    "sequence_pairs": "data/cache/sequence_session_pairs.csv",
    "call_pairs": "data/cache/call_session_pairs.csv",
}
tables = {name: pd.read_csv(registered_artifact(path)) for name, path in paths.items()}
sequence_distances = tables["sequence_distances"]
call_distances = tables["call_distances"]
sequence_model = tables["sequence_model"]
call_model = tables["call_model"]
call_inventory = tables["call_inventory"]
sequence_pairs = tables["sequence_pairs"]
call_pairs = tables["call_pairs"]

base_columns = {
    "comparison_id", "pair_id", "stage", "context",
    "repertoire_a_id", "repertoire_b_id", "metric", "distance",
}
model_columns = base_columns | {"standardized_distance", "stage_centered"}
for name, table, required in (
    ("sequence distances", sequence_distances, base_columns),
    ("call distances", call_distances, base_columns),
    ("sequence model", sequence_model, model_columns),
    ("call model", call_model, model_columns),
):
    if missing := sorted(required.difference(table.columns)):
        raise ValueError(f"{name} is missing canonical columns: {missing}")
    if not np.isfinite(table["distance"]).all() or (table["distance"] < 0).any():
        raise ValueError(f"{name} contains invalid raw distances.")

for name, table in (("sequence model", sequence_model), ("call model", call_model)):
    if not np.isfinite(table["standardized_distance"]).all():
        raise ValueError(f"{name} contains non-finite standardized distances.")
    expected_stage = table["stage"].astype(str).str.lower().map({"before": -0.5, "after": 0.5})
    if expected_stage.isna().any() or not np.allclose(table["stage_centered"], expected_stage):
        raise ValueError(f"{name} has incorrect stage_centered coding.")


def audit_sequence_pair_keys(table: pd.DataFrame, label: str) -> None:
    pair_key = ["comparison_id", "repertoire_a_id", "repertoire_b_id"]
    if missing := sorted(set(pair_key).difference(table.columns)):
        raise ValueError(f"{label} is missing sequence-pair columns: {missing}")
    if missing := sorted(set(pair_key).difference(sequence_pairs.columns)):
        raise ValueError(f"Sequence pair index is missing columns: {missing}")
    indexed = sequence_pairs[pair_key].copy()
    observed = table[pair_key].drop_duplicates()
    if indexed["comparison_id"].duplicated().any():
        raise ValueError("Sequence pair index contains duplicate comparison IDs.")
    if observed["comparison_id"].duplicated().any():
        raise ValueError(f"{label} maps one comparison to multiple repertoire pairs.")
    membership = indexed.merge(
        observed, on=pair_key, how="outer", indicator=True, validate="one_to_one"
    )
    if not membership["_merge"].eq("both").all():
        raise AssertionError(f"{label} does not exactly cover the registered sequence pair index.")


def audit_call_lower_level_pairs(table: pd.DataFrame, label: str) -> None:
    required = {"comparison_id", "repertoire_a_id", "repertoire_b_id", "n_lower_level_pairs"}
    if missing := sorted(required.difference(table.columns)):
        raise ValueError(f"{label} is missing call aggregation columns: {missing}")
    values = pd.to_numeric(table["n_lower_level_pairs"], errors="coerce")
    if values.isna().any() or (values <= 0).any() or not np.equal(values, np.floor(values)).all():
        raise ValueError(f"{label} n_lower_level_pairs must be positive integers.")
    pair_key = ["comparison_id", "repertoire_a_id", "repertoire_b_id"]
    if missing := sorted(set(pair_key).difference(call_pairs.columns)):
        raise ValueError(f"Call pair index is missing columns: {missing}")
    indexed_pairs = call_pairs[pair_key].copy()
    if indexed_pairs["comparison_id"].duplicated().any():
        raise ValueError("Call pair index contains duplicate comparison IDs.")
    observed_pairs = table[pair_key].drop_duplicates()
    if observed_pairs["comparison_id"].duplicated().any():
        raise ValueError(f"{label} maps one comparison to multiple repertoire pairs.")
    membership = indexed_pairs.merge(
        observed_pairs, on=pair_key, how="outer", indicator=True, validate="one_to_one"
    )
    if not membership["_merge"].eq("both").all():
        raise AssertionError(f"{label} does not exactly cover the registered call pair index.")
    if call_inventory["repertoire_id"].duplicated().any():
        raise ValueError("Call inventory contains duplicate repertoire IDs.")
    count_values = pd.to_numeric(call_inventory["n_calls"], errors="coerce")
    if (
        count_values.isna().any()
        or (count_values <= 0).any()
        or not np.equal(count_values, np.floor(count_values)).all()
    ):
        raise ValueError("Call inventory n_calls values must be positive integers.")
    counts = pd.Series(
        count_values.to_numpy(dtype=np.int64), index=call_inventory["repertoire_id"]
    )
    expected = indexed_pairs[["comparison_id"]].copy()
    expected["expected"] = (
        indexed_pairs["repertoire_a_id"].map(counts)
        * indexed_pairs["repertoire_b_id"].map(counts)
    )
    observed = pd.DataFrame({"comparison_id": table["comparison_id"], "observed": values}).drop_duplicates()
    if observed["comparison_id"].duplicated().any():
        raise ValueError(f"{label} metrics disagree on n_lower_level_pairs.")
    product = expected.merge(observed, on="comparison_id", how="outer", validate="one_to_one")
    if product.isna().any().any() or not np.array_equal(
        product["expected"].to_numpy(dtype=np.int64),
        product["observed"].to_numpy(dtype=np.int64),
    ):
        raise AssertionError(f"{label} n_lower_level_pairs != n_calls(a) × n_calls(b).")


audit_sequence_pair_keys(sequence_distances, "sequence distances")
audit_sequence_pair_keys(sequence_model, "sequence model")
audit_call_lower_level_pairs(call_distances, "call distances")
audit_call_lower_level_pairs(call_model, "call model")

{name: table.shape for name, table in tables.items()}

## Verify four rows per comparison and manuscript counts

The expected counts are validation targets taken from the manuscript. Deduplicating by `comparison_id` recovers the number of unique repertoire pairs; the full long-form table must contain exactly four rows per comparison.

In [ ]:
SEQUENCE_METRICS = {
    "transition_probability", "bigram", "phee_repeat", "local_alignment"
}
CALL_METRICS = {"stp", "mfcc", "dtw", "vae"}


def canonical_metric(value: object, structure: str) -> str:
    value = str(value).strip().lower().replace(" ", "_").replace("-", "_")
    aliases = {
        "transition_matrix": "transition_probability",
        "bigram_distribution": "bigram",
        "repeat_distribution": "phee_repeat",
        "spectro_temporal": "stp",
    }
    return aliases.get(value, value)


def normalize_context(value: object) -> str:
    value = str(value).strip().lower().replace("_", "-")
    return "non-partner" if value in {"stranger", "nonpartner", "non-paired"} else value


def audit_long_table(table: pd.DataFrame, structure: str, expected_comparisons: int,
                     expected_partner: int, expected_non_partner: int,
                     expected_rows: int, expected_metrics: set[str]) -> pd.DataFrame:
    canonical = table.copy()
    canonical["metric_canonical"] = canonical["metric"].map(lambda value: canonical_metric(value, structure))
    canonical["context_canonical"] = canonical["context"].map(normalize_context)
    per_comparison = canonical.groupby("comparison_id", observed=True).agg(
        rows=("metric_canonical", "size"), metrics=("metric_canonical", "nunique")
    )
    if not per_comparison["rows"].eq(4).all() or not per_comparison["metrics"].eq(4).all():
        raise AssertionError(f"{structure} does not have exactly four distinct metric rows per comparison.")
    if set(canonical["metric_canonical"]) != expected_metrics:
        raise AssertionError(f"{structure} metric set is not canonical: {set(canonical['metric_canonical'])}")
    unique = canonical.drop_duplicates("comparison_id")
    observed = {
        "unique comparisons": unique["comparison_id"].nunique(),
        "Partner comparisons": int((unique["context_canonical"] == "partner").sum()),
        "Non-partner comparisons": int((unique["context_canonical"] == "non-partner").sum()),
        "long-form rows": len(canonical),
    }
    expected = {
        "unique comparisons": expected_comparisons,
        "Partner comparisons": expected_partner,
        "Non-partner comparisons": expected_non_partner,
        "long-form rows": expected_rows,
    }
    return pd.DataFrame([
        {"structure": structure, "quantity": key, "expected": expected[key],
         "observed": observed[key], "passes": expected[key] == observed[key]}
        for key in expected
    ])


audit = pd.concat([
    audit_long_table(sequence_model, "sequence", 331, 62, 269, 1324, SEQUENCE_METRICS),
    audit_long_table(call_model, "call", 431, 74, 357, 1724, CALL_METRICS),
], ignore_index=True)
display(audit)
if not audit["passes"].all():
    raise AssertionError("Model-table manuscript count audit failed.")

## Verify that model rows preserve the derived distances

The join below is deliberately exact on comparison and metric. It guards against changed pair labels, swapped repertoire identifiers, or a model table assembled from a different distance-cache version.

In [ ]:
def compare_distance_and_model(distance_table: pd.DataFrame, model_table: pd.DataFrame,
                               structure: str) -> dict:
    left = distance_table.copy()
    right = model_table.copy()
    left["metric_canonical"] = left["metric"].map(lambda value: canonical_metric(value, structure))
    right["metric_canonical"] = right["metric"].map(lambda value: canonical_metric(value, structure))
    key = ["comparison_id", "metric_canonical"]
    if left.duplicated(key).any() or right.duplicated(key).any():
        raise ValueError(f"{structure} comparison/metric keys are not unique.")
    merged = left[key + ["distance"]].merge(
        right[key + ["distance"]], on=key, how="outer", suffixes=("_derived", "_model"),
        validate="one_to_one", indicator=True,
    )
    if not merged["_merge"].eq("both").all():
        raise AssertionError(f"{structure} model and distance keys differ.")
    if not np.allclose(merged["distance_derived"], merged["distance_model"], rtol=0, atol=1e-12):
        raise AssertionError(f"{structure} raw distances changed during model-table preparation.")
    return {"structure": structure, "rows_checked": len(merged), "passes": True}


display(pd.DataFrame([
    compare_distance_and_model(sequence_distances, sequence_model, "sequence"),
    compare_distance_and_model(call_distances, call_model, "call"),
]))

## Verify standardization

Each standardized value must equal the raw within-metric population z-score, using `ddof=0`, within an absolute tolerance of `1e-6`. No sample-standard-deviation alternative is accepted.

In [ ]:
standardization_rows = []
for structure, table in (("sequence", sequence_model), ("call", call_model)):
    working = table.assign(
        metric_canonical=table["metric"].map(lambda value: canonical_metric(value, structure))
    )
    for metric, group in working.groupby("metric_canonical", observed=True):
        raw_mean = group["distance"].mean()
        raw_sd_population = group["distance"].std(ddof=0)
        if not np.isfinite(raw_sd_population) or raw_sd_population <= 0:
            raise ValueError(f"{structure}/{metric} has no finite population scale.")
        expected_z = (group["distance"] - raw_mean) / raw_sd_population
        observed_z = group["standardized_distance"]
        max_abs_error = float(np.max(np.abs(observed_z - expected_z)))
        mean = observed_z.mean()
        sd_population = observed_z.std(ddof=0)
        passes = (
            max_abs_error <= 1e-6
            and abs(mean) <= 1e-6
            and abs(sd_population - 1.0) <= 1e-6
        )
        standardization_rows.append({
            "structure": structure, "metric": metric, "n": len(group),
            "ddof": 0, "mean": mean, "population_sd": sd_population,
            "max_abs_error": max_abs_error, "passes": passes,
        })
standardization_audit = pd.DataFrame(standardization_rows)
display(standardization_audit)
if not standardization_audit["passes"].all():
    raise AssertionError("Within-metric standardization audit failed.")

## Figure S1 — analytical workflow

The diagram is a manuscript-reference asset. Its checksum and source revision are validated before display. The quantitative path it summarizes has been checked directly in the cells above.

In [ ]:
figure_path = "results/figures/supplement/figure_s1_workflow.png"
if (ROOT / figure_path).is_file():
    display(Image(filename=str(registered_artifact(figure_path))))
else:
    print(f"Optional manuscript-reference export is not present at {figure_path}.")

## Handoff to modelling

Passing this notebook establishes the exact observations and standardized responses supplied to the two unified Bayesian models. Notebook 05 loads versioned posterior draws, reconstructs the manuscript contrasts, verifies their rounded values, and audits sampling diagnostics.